# QEFM Plotting Notebook

## Imports

In [1]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import xarray as xr
from glob import glob
from pathlib import Path
import yaml
from tqdm import tqdm
import sys
import os
from datetime import datetime

# Import refactored plotting functions
from load_stats import load_regional_stats_data, load_global_stats_data
from plots_regional import create_regional_plots
from plots_global import create_global_plots
from utils import get_stat_dataset_filenames, create_base_patterns

## Constants
Levels: [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50] hPa

Variables:
- 3D variables: [H, Q, T, U, V]
- 2D variables: [P, PS]
- Slice variables: [Q2m, T2m, U10m, V10m, D2m]

Regions:
- NHE: Northern Hemisphere (20°N to 80°N)
- SHE: Southern Hemisphere (80°S to 20°S)
- TRO: Tropical Belt (20°S to 20°N)

Valid dates: any date in May 2024, Dec 2024



In [2]:
# Configure start and end date of experiments
# Dates can be any date in the range of May 2024, or Dec 2024.
START_YEAR = "2024"
START_MONTH = "05"
START_DAY = "01"

END_YEAR = "2024"
END_MONTH = "05"
END_DAY = "31"

DATE_STR = f"{START_YEAR}{START_MONTH}{START_DAY}-{END_YEAR}{END_MONTH}{END_DAY}"

# We will look for stats in this directory; stats are stored in netcdf files
ROOT_DIR = "outputs"  # Where to look for stats
FORECAST_MODEL = "Prithvi"  # Choose from 'AIFS' or 'Prithvi'

# Will only plot these variables across 3D and 2D collections
VARS_TO_PLOT = ["T"]

# Will only create level plots using these levels (zonal plots will still be created with all levels)
LEVELS_TO_PLOT = [850, 500, 200]

# Will only plot leads from this list; forecasts were made for up to 7 days
# Leads are in hours
LEADS_TO_PLOT = [24, 72, 120]

# Percentiles to use for plotting color limits
PERCENTILES = [90, 65, 65]

# Choose from among [NHE, TRO, SHE]
# Regional stats will make plots for only these regions
REGIONS_TO_PLOT = ["NHE", "SHE", "TRO"]

## Get filenames of statistics netCDFs
These have already been created.

In [3]:
# Create list of all statistics filenames across all experiments
exp_patterns = create_base_patterns(FORECAST_MODEL)
all_stat_filenames = get_stat_dataset_filenames(exp_patterns, DATE_STR)
regional_stat_filenames = all_stat_filenames["regional"]
global_stat_filenames = all_stat_filenames["global"]

# There should be 1 file per experiment in each of regional, global filename lists
regional_file_str = "\n".join(map(str, regional_stat_filenames))
global_file_str = "\n".join(map(str, global_stat_filenames))
print(f"Regional stat files:\n{regional_file_str}")
print(f"Global stat files:\n{global_file_str}")

may_2024
Regional stat files:
['s3://enw-04241552-kx1nks-shared/data/stats/may_2024/stats_regional_f5295fp_GEOSFP_MERRA2_20240501-20240531_len7d_int12h_spc1d_91x144.nc4']
['s3://enw-04241552-kx1nks-shared/data/stats/may_2024/stats_regional_MERRA2_MERRA2_MERRA2_20240501-20240531_len7d_int12h_spc1d_91x144.nc4']
['s3://enw-04241552-kx1nks-shared/data/stats/may_2024/stats_regional_Prithvi_GEOSFP_MERRA2_20240501-20240531_len7d_int12h_spc1d_91x144.nc4']
['s3://enw-04241552-kx1nks-shared/data/stats/may_2024/stats_regional_Prithvi_MERRA2_MERRA2_20240501-20240531_len7d_int12h_spc1d_91x144.nc4']
Global stat files:
['s3://enw-04241552-kx1nks-shared/data/stats/may_2024/stats_global_f5295fp_GEOSFP_MERRA2_20240501-20240531_len7d_int12h_spc1d_91x144.nc4']
['s3://enw-04241552-kx1nks-shared/data/stats/may_2024/stats_global_MERRA2_MERRA2_MERRA2_20240501-20240531_len7d_int12h_spc1d_91x144.nc4']
['s3://enw-04241552-kx1nks-shared/data/stats/may_2024/stats_global_Prithvi_GEOSFP_MERRA2_20240501-20240531_len7

## Load important data from stats netcdf

### Regional

In [4]:
regional_data = load_regional_stats_data(
    regional_stat_filenames, 
    plot_RMS_decomp=False, 
    verbose=False, 
    vars_to_fetch=VARS_TO_PLOT
)

Starting loading of regional stat data from netcdf...
Found 4 experiment files
[['s3://enw-04241552-kx1nks-shared/data/stats/may_2024/stats_regional_f5295fp_GEOSFP_MERRA2_20240501-20240531_len7d_int12h_spc1d_91x144.nc4'], ['s3://enw-04241552-kx1nks-shared/data/stats/may_2024/stats_regional_MERRA2_MERRA2_MERRA2_20240501-20240531_len7d_int12h_spc1d_91x144.nc4'], ['s3://enw-04241552-kx1nks-shared/data/stats/may_2024/stats_regional_Prithvi_GEOSFP_MERRA2_20240501-20240531_len7d_int12h_spc1d_91x144.nc4'], ['s3://enw-04241552-kx1nks-shared/data/stats/may_2024/stats_regional_Prithvi_MERRA2_MERRA2_20240501-20240531_len7d_int12h_spc1d_91x144.nc4']]
Loading single file for exp0...


/home/sagemaker-user/ESA-NASA-Workshop-2026/Day 4/Track 3/Weather-Foundation-Models/ESA-NASA-workshop-2026/Track 3/Weather-Foundation-Models/.venv/lib/python3.11/site-packages/fsspec/registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


Dataset contains 31 init dates from 20240501_00 to 20240531_00
Loading single file for exp1...
Dataset contains 31 init dates from 20240501_00 to 20240531_00
Loading single file for exp2...
Dataset contains 31 init dates from 20240501_00 to 20240531_00
Loading single file for exp3...
Dataset contains 31 init dates from 20240501_00 to 20240531_00
✓ All experiments have identical dates - no filtering needed


### Global

In [5]:
global_data_comp = load_global_stats_data(
    global_stat_filenames, 
    process='comp', 
    load_raw=True, 
    verbose=True, 
    vars_to_fetch=VARS_TO_PLOT
)

Loading 4 experiment(s) for comp process...

--- Experiment 0 ---
Loading single file for exp0...
Dataset contains 31 init dates from 20240501_00 to 20240531_00

--- Experiment 1 ---
Loading single file for exp1...
Dataset contains 31 init dates from 20240501_00 to 20240531_00

--- Experiment 2 ---
Loading single file for exp2...
Dataset contains 31 init dates from 20240501_00 to 20240531_00

--- Experiment 3 ---
Loading single file for exp3...
Dataset contains 31 init dates from 20240501_00 to 20240531_00

=== Experiment Summary ===
  exp0: f5295fp(F) / GEOSFP(A) / MERRA2(C)
  exp1: MERRA2(F) / MERRA2(A) / MERRA2(C)
  exp2: Prithvi(F) / GEOSFP(A) / MERRA2(C)
  exp3: Prithvi(F) / MERRA2(A) / MERRA2(C)

=== Validating Dates (Strict) ===
  All experiments have identical 31 dates (20240501_00 to 20240531_00) ✓

=== Finding Common Variables ===
  Before filtering, common vars:
  de3d: ['Q', 'T', 'U', 'V']
  sl2d: ['D2m', 'T2m', 'U10m', 'V10m']
  Stats: ['acorr', 'acorr_glo', 'f', 'f_glo', 

# Stat Plotting

## Make output dir

In [7]:
output_dir = Path(f"outputs/Prithvi_viridis")
output_dir.mkdir(exist_ok=True, parents=True)

## Regional stat plotting

In [8]:
# Creates only per-level plots of ACC/RMSE
create_regional_plots(
    regional_data,
    vars_to_plot=VARS_TO_PLOT, 
    levels_to_plot=LEVELS_TO_PLOT,
    leads_to_plot=None,  # Lead filtering is nebulous
    regions_to_plot=REGIONS_TO_PLOT,
    stats_to_plot=[0, 1],  # ACC, RMSE only
    plots_dir=output_dir
)


=== Running Plot Mode ===

Discovered experiments: [0, 1, 2, 3]
Filtered regions: ['NHE', 'TRO', 'SHE'].
Active collections after filtering: ['de3d']
Filtered variables: ['T']
Filtering statistics to first 2 (indices: [0, 1])
Models: {0: 'f5295fp_GEOSFP', 1: 'MERRA2_MERRA2', 2: 'Prithvi_GEOSFP', 3: 'Prithvi_MERRA2'}
Created corcmp.rc

    Variable indices: [0 0 0 0 0 0 0 0 0 0 0 0 0], Level indices: [ 0  1  2  3  4  5  6  7  8  9 10 11 12]
    Variable indices: [0 0 0 0 0 0 0 0 0 0 0 0 0], Level indices: [ 0  1  2  3  4  5  6  7  8  9 10 11 12]
    Variable indices: [0 0 0 0 0 0 0 0 0 0 0 0 0], Level indices: [ 0  1  2  3  4  5  6  7  8  9 10 11 12]
    Variable indices: [0 0 0 0 0 0 0 0 0 0 0 0 0], Level indices: [ 0  1  2  3  4  5  6  7  8  9 10 11 12]
    Variable indices: [0 0 0 0 0 0 0 0 0 0 0 0 0], Level indices: [ 0  1  2  3  4  5  6  7  8  9 10 11 12]
    Variable indices: [0 0 0 0 0 0 0 0 0 0 0 0 0], Level indices: [ 0  1  2  3  4  5  6  7  8  9 10 11 12]
    Variable indices

## Global stat plotting

In [9]:
create_global_plots(
    global_data_comp,
    vars_to_plot=VARS_TO_PLOT, 
    levels_to_plot=LEVELS_TO_PLOT,
    leads_to_plot=LEADS_TO_PLOT,
    exps_to_comp=[[3, 1], [2, 0]],  # Compare models across 2 rows
    colormap_key="ryb",
    gif_frame_duration=1000,
    output_dir=output_dir
)


GLOBAL STATS PLOTTING DRIVER - COMP MODE



/home/sagemaker-user/ESA-NASA-Workshop-2026/Day 4/Track 3/Weather-Foundation-Models/ESA-NASA-workshop-2026/Track 3/Weather-Foundation-Models/.venv/lib/python3.11/site-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/110m_physical/ne_110m_coastline.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


Using hard-coded colormap limits:
  T: Forecast_diff=±5.0, ACC_raw=[0,1.0], RMSE_raw=[0,10.0]
  U: Forecast_diff=±2.0, ACC_raw=[0,1.0], RMSE_raw=[0,5.0]
  V: Forecast_diff=±2.0, ACC_raw=[0,1.0], RMSE_raw=[0,5.0]
  Z: Forecast_diff=±50.0, ACC_raw=[0,1.0], RMSE_raw=[0,100.0]
  Q: Forecast_diff=±1.0, ACC_raw=[0,1.0], RMSE_raw=[0,2.0]

Extracting data from dictionary...
Available experiments in data: [0, 1, 2, 3]
Comparison mode: will compare 4 experiments to baseline (exp 0)

Model mapping:
  Experiment 0: GEOSFP_GEOSFP
  Experiment 1: MERRA2_MERRA2
  Experiment 2: Prithvi_GEOSFP
  Experiment 3: Prithvi_MERRA2
Received a list of experiments to compare.
For comparison [3, 1], will compare Prithvi_MERRA2 to MERRA2_MERRA2
For comparison [2, 0], will compare Prithvi_GEOSFP to GEOSFP_GEOSFP

Filtering variables to: ['T']
Filtered fvars: {'de3d': ['T'], 'de2d': [], 'sl2d': [], 'ae2d': []}
Filtering lead times to: [24, 72, 120]

GENERATING COMP PLOTS

  Collection: de3d
    Making comp plots for